## A Spectabular Model of a Simple Elevator

A simple elevator serves three floors (F1, F2, F3) implementing an elevator algorithm: the elevator travels in one direction, serving all requests in that direction, then reverses or stops.

**State:** current floor, travel direction, four exterior call buttons (up at F1, up/down at F2, down at F3), and three interior cabin buttons.

**Events (10 total):**
- 4 exterior button presses: `Press1U`, `Press2U`, `Press2D`, `Press3D`
- 3 cabin button presses: `PressC1`, `PressC2`, `PressC3`
- 3 floor arrivals: `ArrF1`, `ArrF2`, `ArrF3`

Arrival events are enabled only when the elevator is at an adjacent floor moving in the correct direction.

In [1]:
%run "spectabular.ipynb"

### State Variables

The floor and direction are enumeration types. Four Boolean variables model the exterior call buttons (one per button position) and three model the cabin buttons (one per floor).

In [2]:
EnumType('Floor', 'F1', 'F2', 'F3')
Floor('floor')

EnumType('Dir', 'Up', 'Down', 'Idle')
Dir('dir')

Bool('b1u')   # exterior: floor 1, up
Bool('b2u')   # exterior: floor 2, up
Bool('b2d')   # exterior: floor 2, down
Bool('b3d')   # exterior: floor 3, down
Bool('c1')    # cabin: floor 1
Bool('c2')    # cabin: floor 2
Bool('c3')    # cabin: floor 3

### Event Type

In [3]:
Enum('ev', 'Press1U', 'Press2U', 'Press2D', 'Press3D',
     'PressC1', 'PressC2', 'PressC3',
     'ArrF1', 'ArrF2', 'ArrF3')

### Invariant

The elevator must not attempt to travel beyond the building boundaries: it cannot be moving up while at the top floor, nor moving down while at the bottom floor.

In [4]:
INV1 = \
    Implies(dir == Up, floor != F3); INV1

Implies(dir == Up, floor != F3)

In [5]:
INV2 = \
    Implies(dir == Down, floor != F1); INV2

Implies(dir == Down, floor != F1)

In [6]:
INV = INV1 & INV2; INV

And(Implies(dir == Up, floor != F3),
    Implies(dir == Down, floor != F1))

### Button Press Events

When a button is pressed, that button becomes active. If the elevator is already moving (`dir ≠ Idle`), the direction is preserved. If the elevator is idle, it starts moving toward the requested floor.

Each button targets a floor. The direction when idle is determined by the relative position of the current floor and the target:
- Target above current floor: `dirʹ = Up`
- Target is current floor: `dirʹ = Idle`
- Target below current floor:`dirʹ = Down`

#### Exterior Buttons Targeting Floor 1

In [7]:
press1UEvent = \
    VectorTable(
        ((dir == Idle, (floor == F1, floor != F1)), dir != Idle),
        (dirʹ,   Idle, Down, dir),
        (b1uʹ,   True, True, True),
        (b2uʹ,   b2u,  b2u,  b2u),
        (b2dʹ,   b2d,  b2d,  b2d),
        (b3dʹ,   b3d,  b3d,  b3d),
        (c1ʹ,    c1,   c1,   c1),
        (c2ʹ,    c2,   c2,   c2),
        (c3ʹ,    c3,   c3,   c3),
        (floorʹ, floor, floor, floor)); press1UEvent

#### Exterior Buttons Targeting Floor 2

In [8]:
press2UEvent = \
    VectorTable(
        ((dir == Idle, (floor == F1, floor == F2, floor == F3)), dir != Idle),
        (dirʹ,   Up,   Idle, Down, dir),
        (b1uʹ,   b1u,  b1u,  b1u,  b1u),
        (b2uʹ,   True, True, True, True),
        (b2dʹ,   b2d,  b2d,  b2d,  b2d),
        (b3dʹ,   b3d,  b3d,  b3d,  b3d),
        (c1ʹ,    c1,   c1,   c1,   c1),
        (c2ʹ,    c2,   c2,   c2,   c2),
        (c3ʹ,    c3,   c3,   c3,   c3),
        (floorʹ, floor, floor, floor, floor)); press2UEvent

In [9]:
press2DEvent = \
    VectorTable(
        ((dir == Idle, (floor == F1, floor == F2, floor == F3)), dir != Idle),
        (dirʹ,   Up,   Idle, Down, dir),
        (b1uʹ,   b1u,  b1u,  b1u,  b1u),
        (b2uʹ,   b2u,  b2u,  b2u,  b2u),
        (b2dʹ,   True, True, True, True),
        (b3dʹ,   b3d,  b3d,  b3d,  b3d),
        (c1ʹ,    c1,   c1,   c1,   c1),
        (c2ʹ,    c2,   c2,   c2,   c2),
        (c3ʹ,    c3,   c3,   c3,   c3),
        (floorʹ, floor, floor, floor, floor)); press2DEvent

#### Exterior Button Targeting Floor 3

In [10]:
press3DEvent = \
    VectorTable(
        ((dir == Idle, (floor == F3, floor != F3)), dir != Idle),
        (dirʹ,   Idle, Up,   dir),
        (b1uʹ,   b1u,  b1u,  b1u),
        (b2uʹ,   b2u,  b2u,  b2u),
        (b2dʹ,   b2d,  b2d,  b2d),
        (b3dʹ,   True, True, True),
        (c1ʹ,    c1,   c1,   c1),
        (c2ʹ,    c2,   c2,   c2),
        (c3ʹ,    c3,   c3,   c3),
        (floorʹ, floor, floor, floor)); press3DEvent

#### Cabin Buttons

Cabin buttons follow the same direction logic as exterior buttons targeting the corresponding floor. Only the activated button variable differs.

In [11]:
pressC1Event = \
    VectorTable(
        ((dir == Idle, (floor == F1, floor != F1)), dir != Idle),
        (dirʹ,   Idle, Down, dir),
        (b1uʹ,   b1u,  b1u,  b1u),
        (b2uʹ,   b2u,  b2u,  b2u),
        (b2dʹ,   b2d,  b2d,  b2d),
        (b3dʹ,   b3d,  b3d,  b3d),
        (c1ʹ,    True, True, True),
        (c2ʹ,    c2,   c2,   c2),
        (c3ʹ,    c3,   c3,   c3),
        (floorʹ, floor, floor, floor)); pressC1Event

In [12]:
pressC2Event = \
    VectorTable(
        ((dir == Idle, (floor == F1, floor == F2, floor == F3)), dir != Idle),
        (dirʹ,   Up,   Idle, Down, dir),
        (b1uʹ,   b1u,  b1u,  b1u,  b1u),
        (b2uʹ,   b2u,  b2u,  b2u,  b2u),
        (b2dʹ,   b2d,  b2d,  b2d,  b2d),
        (b3dʹ,   b3d,  b3d,  b3d,  b3d),
        (c1ʹ,    c1,   c1,   c1,   c1),
        (c2ʹ,    True, True, True, True),
        (c3ʹ,    c3,   c3,   c3,   c3),
        (floorʹ, floor, floor, floor, floor)); pressC2Event

In [13]:
pressC3Event = \
    VectorTable(
        ((dir == Idle, (floor == F3, floor != F3)), dir != Idle),
        (dirʹ,   Idle, Up,   dir),
        (b1uʹ,   b1u,  b1u,  b1u),
        (b2uʹ,   b2u,  b2u,  b2u),
        (b2dʹ,   b2d,  b2d,  b2d),
        (b3dʹ,   b3d,  b3d,  b3d),
        (c1ʹ,    c1,   c1,   c1),
        (c2ʹ,    c2,   c2,   c2),
        (c3ʹ,    True, True, True),
        (floorʹ, floor, floor, floor)); pressC3Event

#### Combined Button Press Event

In [14]:
buttonEvent = \
    Table(
        (ev == Press1U, ev == Press2U, ev == Press2D, ev == Press3D,
         ev == PressC1, ev == PressC2, ev == PressC3),
        ("press1UEvent", "press2UEvent", "press2DEvent", "press3DEvent",
         "pressC1Event", "pressC2Event", "pressC3Event")); buttonEvent

In [15]:
assert buttonEvent.disjoint

### Arrival Events

An arrival event fires when the elevator reaches an adjacent floor while moving in that direction. On arrival, the floor is updated, call and cabin buttons at the floor are cleared, and the algorithm determines the next direction:

1. Continue in the current direction if there are requests ahead.
2. Reverse if there are no requests ahead but requests behind (or at the current floor in the reverse direction).
3. Stop if there are no requests in either direction.

When reversing at floor 2, passengers waiting in the reverse direction are also picked up (their button is cleared).

These varibles capture "any request above/below" a given floor, excluding buttons that are cleared on arrival at that floor.

In [16]:
reqAboveF1 = b2u | b2d | b3d | c2 | c3   # requests above F1
reqBelowF3 = b1u | b2u | b2d | c1 | c2       # requests below F3
reqAboveF2 = b3d | c3                          # requests above F2
reqBelowF2 = b1u | c1                          # requests below F2

#### Arrival at Floor 1

Enabled when: `floor = F2 ∧ dir = Down`. Clears `b1u` and `c1`. If there are requests above, the elevator reverses to `Up`; otherwise it stops.

In [17]:
arriveF1Event = \
    VectorTable(
        (reqAboveF1, ~reqAboveF1),
        (floorʹ, F1,    F1),
        (dirʹ,   Up,    Idle),
        (b1uʹ,   False, False),
        (b2uʹ,   b2u,   b2u),
        (b2dʹ,   b2d,   b2d),
        (b3dʹ,   b3d,   b3d),
        (c1ʹ,    False, False),
        (c2ʹ,    c2,    c2),
        (c3ʹ,    c3,    c3)); arriveF1Event

#### Arrival at Floor 2 from Below

Enabled when: `floor = F1 ∧ dir = Up`. Clears `b2u` and `c2`. Following the elevator algorithm: continue up if requests above; reverse to down (also clearing `b2d`) if requests below or at F2 going down; otherwise stop.

In [18]:
arriveF2UpEvent = \
    VectorTable(
        (reqAboveF2, ~reqAboveF2 & (reqBelowF2 | b2d), ~reqAboveF2 & ~reqBelowF2 & ~b2d),
        (floorʹ, F2,   F2,    F2),
        (dirʹ,   Up,   Down,  Idle),
        (b1uʹ,   b1u,  b1u,   b1u),
        (b2uʹ,   False, False, False),
        (b2dʹ,   b2d,  False, False),
        (b3dʹ,   b3d,  b3d,   b3d),
        (c1ʹ,    c1,   c1,    c1),
        (c2ʹ,    False, False, False),
        (c3ʹ,    c3,   c3,    c3)); arriveF2UpEvent

#### Arrival at Floor 2 from Above

Enabled when: `floor = F3 ∧ dir = Down`. Clears `b2d` and `c2`. Continue down if requests below; reverse to up (also clearing `b2u`) if requests above or at F2 going up; otherwise stop.

In [19]:
arriveF2DownEvent = \
    VectorTable(
        (reqBelowF2, ~reqBelowF2 & (reqAboveF2 | b2u), ~reqBelowF2 & ~reqAboveF2 & ~b2u),
        (floorʹ, F2,    F2,   F2),
        (dirʹ,   Down,  Up,   Idle),
        (b1uʹ,   b1u,   b1u,  b1u),
        (b2uʹ,   b2u,   False, False),
        (b2dʹ,   False, False, False),
        (b3dʹ,   b3d,   b3d,  b3d),
        (c1ʹ,    c1,    c1,   c1),
        (c2ʹ,    False, False, False),
        (c3ʹ,    c3,    c3,   c3)); arriveF2DownEvent

#### Arrival at Floor 3

Enabled when: `floor = F2 ∧ dir = Up`. Clears `b3d` and `c3`. If there are requests below, the elevator reverses to `Down`; otherwise it stops.

In [20]:
arriveF3Event = \
    VectorTable(
        (reqBelowF3, ~reqBelowF3),
        (floorʹ, F3,    F3),
        (dirʹ,   Down,  Idle),
        (b1uʹ,   b1u,   b1u),
        (b2uʹ,   b2u,   b2u),
        (b2dʹ,   b2d,   b2d),
        (b3dʹ,   False, False),
        (c1ʹ,    c1,    c1),
        (c2ʹ,    c2,    c2),
        (c3ʹ,    False, False)); arriveF3Event

#### Combined Arrival Event

In [21]:
arrivalEvent = \
    Table(
        ((ev == ArrF1) & (floor == F2) & (dir == Down),
         (ev == ArrF2) & (floor == F1) & (dir == Up),
         (ev == ArrF2) & (floor == F3) & (dir == Down),
         (ev == ArrF3) & (floor == F2) & (dir == Up)),
        ("arriveF1Event", "arriveF2UpEvent",
         "arriveF2DownEvent", "arriveF3Event")); arrivalEvent

In [22]:
assert arrivalEvent.disjoint

In [29]:
espec = arrivalEvent | buttonEvent; espec

In [1]:
#valid(Dom(buttonEvent))

### Invariant Preservation

Button press event: When already moving, the direction is unchanged. When idle, the direction is set based on relative floor positions, which always respects the building boundaries:

In [24]:
buttonEventCorrect = Correct("INV", "buttonEvent", "INV"); buttonEventCorrect

In [25]:
assert valid(buttonEventCorrect)

Arrival events: At terminal floors (F1, F3), the elevator can only reverse or stop. At the middle floor (F2), any direction is safe:

In [26]:
arrivalEventCorrect = Correct("INV", "arrivalEvent", "INV"); arrivalEventCorrect

In [27]:
assert valid(arrivalEventCorrect)